In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit
src exists: True


In [3]:
import sys
import sentence_transformers

print("Python:", sys.executable)
print("sentence-transformers:", sentence_transformers.__version__)

Python: /home/sotatek/Projects/Tools/Notebook/.venv/bin/python
sentence-transformers: 6.0.1


In [4]:
import faiss

print("FAISS:", faiss.__version__)

FAISS: 1.15.0


In [5]:
from src.tools.retriever.retriever import Retriever
from src.tools.reranker.reranker import Reranker

print("Retriever import: OK")
print("Reranker import: OK")

Retriever import: OK
Reranker import: OK


In [8]:
retriever = Retriever(
    index_path=PROJECT_ROOT / "virtual_database" / "index.faiss",
    contexts_path=PROJECT_ROOT / "virtual_database" / "contexts.json",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

print("Retriever: OK")
print("FAISS vectors:", retriever.index.ntotal)
print("Contexts:", len(retriever.contexts))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1973.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever: OK
FAISS vectors: 1003
Contexts: 1003


In [9]:
reranker = Reranker(
    model_name="cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)

print("Reranker: OK")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1611.99it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker: OK


In [12]:
TEST_FILE = "../dataset_test/test.json"
CONTEXT_FILE = "../virtual_database/context.json"

In [13]:
query = "Đan Trường là ai?"

result = retriever.retrieve(
    query,
    top_k=5
)

print("Số documents:", len(result["documents"]))

for i, doc in enumerate(result["documents"], start=1):
    print(f"\n--- Rank {i} ---")
    print("ID:", doc["id"])
    print("Content:", doc["content"][:300])

Số documents: 5

--- Rank 1 ---
ID: doc_355
Content: Mỗi năm có từ 1 đến 3 ngày thứ 6 ngày 13 .

--- Rank 2 ---
ID: doc_531
Content: Hiện tại anh thi đấu cho Power F.C..

--- Rank 3 ---
ID: doc_529
Content: Địa lý châu Á có thể coi là phức tạp và đa dạng nhất trong số 5 châu lục trên mặt đất .

--- Rank 4 ---
ID: doc_009
Content: Đan Trường ( sinh năm 1919 ) , tên thật Ngô Đức Vân Quỳnh , là một nhạc sĩ Việt Nam , tác giả nhạc phẩm " Trách người đi " .

--- Rank 5 ---
ID: doc_039
Content: Châu Á chiếm 8.6% tổng diện tích bề mặt Trái Đất ( chiếm 29.9% diện tích mặt đất ) và có 4 tỉ người , chiếm 60% dân số hiện nay của thế giới .


In [14]:
query = "Đan Trường là ai?"

retrieved = retriever.retrieve(
    query,
    top_k=5
)

reranked = reranker.rerank(
    query,
    retrieved["documents"],
    top_k=3
)

print("=== RERANKED RESULTS ===")

for item in reranked["documents"]:
    doc = item["document"]

    print(f"\nRank: {item['rank']}")
    print(f"ID: {doc['id']}")
    print(f"Score: {item['score']:.4f}")
    print(f"Content: {doc['content'][:300]}")

=== RERANKED RESULTS ===

Rank: 1
ID: doc_009
Score: 10.4412
Content: Đan Trường ( sinh năm 1919 ) , tên thật Ngô Đức Vân Quỳnh , là một nhạc sĩ Việt Nam , tác giả nhạc phẩm " Trách người đi " .

Rank: 2
ID: doc_531
Score: -3.0525
Content: Hiện tại anh thi đấu cho Power F.C..

Rank: 3
ID: doc_355
Score: -4.5057
Content: Mỗi năm có từ 1 đến 3 ngày thứ 6 ngày 13 .


In [18]:
import pprint

print("Số lượng test:", len(test_data))

for i, item in enumerate(test_data[:3]):
    print("=" * 80)
    print(f"ITEM {i}")
    print("Keys:", item.keys())
    pprint.pprint(item, width=120)

Số lượng test: 1003
ITEM 0
Keys: dict_keys(['id', 'question', 'answer'])
{'answer': 'đúng',
 'id': '7407e25ef6180902d432faa084eb27c4cf0b25ba22c0df20bc5fbd6e2968ba4a',
 'question': 'Đội bóng Boca Juniors mà Diego Maradona đã chơi có trụ sở tại La Boca, gần Buenos Aires phải không?'}
ITEM 1
Keys: dict_keys(['id', 'question', 'answer'])
{'answer': 'đúng',
 'id': '43a22220b201689db9daaff86d93c3901faff5acb2d6aa8c5bb90d180c5695a1',
 'question': 'Karl-Heinz Rummenigge nhỏ tuổi hơn Dino Zoff phải không?'}
ITEM 2
Keys: dict_keys(['id', 'question', 'answer'])
{'answer': '4',
 'id': '8a8f3ce06c995cf6396eb10955478729d9d1673b58d42d0a2b3a58d52ecbb7a3',
 'question': 'Tiểu bang mà nhân vật Jason Voorhees sống xếp thứ mấy trong danh sách những tiểu bang nhỏ nhất?'}


In [19]:
import json

CONTEXTS_FILE = PROJECT_ROOT / "virtual_database" / "contexts.json"

with open(CONTEXTS_FILE, "r", encoding="utf-8") as f:
    contexts = json.load(f)

print("Số lượng contexts:", len(contexts))

for i, context in enumerate(contexts[:5]):
    print("=" * 80)
    print(f"CONTEXT {i}")
    print(context[:1000] if isinstance(context, str) else context)

Số lượng contexts: 1003
CONTEXT 0
Ông đã chơi cho Argentinos Juniors, Boca Juniors, Barcelona, Napoli, Sevilla và Newell's Old Boys trong suốt sự nghiệp câu lạc bộ của mình, và nổi tiếng nhất là trong thời gian ở Napoli, nơi ông giành được nhiều danh hiệu. Club Atlético Boca Juniors là 1 câu lạc bộ bóng đá chuyên nghiệp của Argentina, có trụ sở tại La Boca, gần Buenos Aires.
CONTEXT 1
Karl-Heinz "Kalle" Rummenigge (sinh 25 tháng 9 năm 1955 tại Lippstadt, Tây Đức) là cựu cầu thủ bóng đá người Đức và hiện tại đang làm CEO của Bayern München. Dino Zoff (sinh ngày 28 tháng 2 năm 1942) là cựu cầu thủ bóng đá Ý, chơi ở vị trí thủ môn và là cầu thủ lớn tuổi nhất trong lịch sử từng vô địch World Cup, vào năm ông 40 tuổi khi đang là đội trưởng đội tuyển Ý tại World Cup 1982 tổ chức tại Tây Ban Nha.
CONTEXT 2
Jason sống ở khu cắm trại hồ Crystal ở hạt Wessex bang New Jersey nước Mỹ, hắn giết bất cứ ai đến cắm trại nơi đây. New Jersey là tiểu bang xếp thứ 4 trong danh sách những tiểu bang nhỏ nhấ

In [20]:
print("Test samples :", len(test_data))
print("Contexts     :", len(contexts))
print("FAISS vectors:", retriever.index.ntotal)

assert len(test_data) == 1003
assert len(contexts) == 1003
assert retriever.index.ntotal == 1003

print("\n✅ Test ↔ Contexts ↔ FAISS có cùng 1003 records")

Test samples : 1003
Contexts     : 1003
FAISS vectors: 1003

✅ Test ↔ Contexts ↔ FAISS có cùng 1003 records


In [21]:
for i in range(5):
    question = test_data[i]["question"]
    relevant_id = f"doc_{i:03d}"
    relevant_context = contexts[i]

    print("=" * 80)
    print("Index       :", i)
    print("Question    :", question)
    print("Relevant ID :", relevant_id)
    print("Context     :", relevant_context[:300])

Index       : 0
Question    : Đội bóng Boca Juniors mà Diego Maradona đã chơi có trụ sở tại La Boca, gần Buenos Aires phải không?
Relevant ID : doc_000
Context     : Ông đã chơi cho Argentinos Juniors, Boca Juniors, Barcelona, Napoli, Sevilla và Newell's Old Boys trong suốt sự nghiệp câu lạc bộ của mình, và nổi tiếng nhất là trong thời gian ở Napoli, nơi ông giành được nhiều danh hiệu. Club Atlético Boca Juniors là 1 câu lạc bộ bóng đá chuyên nghiệp của Argentin
Index       : 1
Question    : Karl-Heinz Rummenigge nhỏ tuổi hơn Dino Zoff phải không?
Relevant ID : doc_001
Context     : Karl-Heinz "Kalle" Rummenigge (sinh 25 tháng 9 năm 1955 tại Lippstadt, Tây Đức) là cựu cầu thủ bóng đá người Đức và hiện tại đang làm CEO của Bayern München. Dino Zoff (sinh ngày 28 tháng 2 năm 1942) là cựu cầu thủ bóng đá Ý, chơi ở vị trí thủ môn và là cầu thủ lớn tuổi nhất trong lịch sử từng vô đị
Index       : 2
Question    : Tiểu bang mà nhân vật Jason Voorhees sống xếp thứ mấy trong danh sách những tiể

In [22]:
import time

CONFIGS = [
    {"retriever_top_k": 5, "reranker_top_n": 3},
    {"retriever_top_k": 10, "reranker_top_n": 3},
    {"retriever_top_k": 20, "reranker_top_n": 5},
    {"retriever_top_k": 50, "reranker_top_n": 5},
]

# Lấy 5 câu đầu để smoke test
smoke_data = test_data[:5]

for config in CONFIGS:
    top_k = config["retriever_top_k"]
    top_n = config["reranker_top_n"]

    print("\n" + "=" * 80)
    print(f"STRATEGY: Top-{top_k} → Top-{top_n}")
    print("=" * 80)

    for i, item in enumerate(smoke_data):

        query = item["question"]
        relevant_id = f"doc_{i:03d}"

        # -------------------------
        # 1. Retrieval
        # -------------------------
        start = time.perf_counter()

        retrieval_result = retriever.retrieve(
            query,
            top_k=top_k
        )

        retrieval_ms = (time.perf_counter() - start) * 1000

        candidates = retrieval_result["documents"]

        # -------------------------
        # 2. Reranking
        # -------------------------
        start = time.perf_counter()

        rerank_result = reranker.rerank(
            query,
            candidates,
            top_k=top_n
        )

        reranker_ms = (time.perf_counter() - start) * 1000

        final_docs = rerank_result["documents"]

        # -------------------------
        # 3. Check relevant document
        # -------------------------
        relevant_rank = None

        for result in final_docs:
            if result["document"]["id"] == relevant_id:
                relevant_rank = result["rank"]
                break

        hit = relevant_rank is not None
        rr = 1 / relevant_rank if relevant_rank else 0

        # -------------------------
        # 4. Print
        # -------------------------
        print(f"\nQuery {i + 1}")
        print(f"Relevant ID : {relevant_id}")
        print(f"Candidates  : {len(candidates)}")
        print(f"Final docs  : {len(final_docs)}")
        print(f"Relevant rank: {relevant_rank}")
        print(f"Hit         : {hit}")
        print(f"RR          : {rr:.3f}")
        print(f"Retrieval   : {retrieval_ms:.2f} ms")
        print(f"Reranker    : {reranker_ms:.2f} ms")

        # In ra các document sau reranking
        print("Final ranking:")

        for result in final_docs:
            doc_id = result["document"]["id"]
            score = result["score"]
            rank = result["rank"]

            marker = " <-- RELEVANT" if doc_id == relevant_id else ""

            print(
                f"  Rank {rank}: "
                f"{doc_id} | "
                f"score={score:.4f}"
                f"{marker}"
            )


STRATEGY: Top-5 → Top-3

Query 1
Relevant ID : doc_000
Candidates  : 5
Final docs  : 3
Relevant rank: 1
Hit         : True
RR          : 1.000
Retrieval   : 12.22 ms
Reranker    : 6.51 ms
Final ranking:
  Rank 1: doc_000 | score=9.1959 <-- RELEVANT
  Rank 2: doc_577 | score=-6.3292
  Rank 3: doc_365 | score=-6.8362

Query 2
Relevant ID : doc_001
Candidates  : 5
Final docs  : 3
Relevant rank: 1
Hit         : True
RR          : 1.000
Retrieval   : 2.37 ms
Reranker    : 5.23 ms
Final ranking:
  Rank 1: doc_001 | score=9.5994 <-- RELEVANT
  Rank 2: doc_656 | score=-6.5645
  Rank 3: doc_377 | score=-6.9775

Query 3
Relevant ID : doc_002
Candidates  : 5
Final docs  : 3
Relevant rank: 1
Hit         : True
RR          : 1.000
Retrieval   : 2.36 ms
Reranker    : 5.47 ms
Final ranking:
  Rank 1: doc_002 | score=6.6404 <-- RELEVANT
  Rank 2: doc_676 | score=-4.3968
  Rank 3: doc_735 | score=-4.5393

Query 4
Relevant ID : doc_003
Candidates  : 5
Final docs  : 3
Relevant rank: None
Hit         : F

In [24]:
import time
import pandas as pd

CONFIGS = [
    {"retriever_top_k": 5, "reranker_top_n": 3},
    {"retriever_top_k": 10, "reranker_top_n": 3},
    {"retriever_top_k": 20, "reranker_top_n": 5},
    {"retriever_top_k": 50, "reranker_top_n": 5},
]


def evaluate_config(
    test_data,
    retriever,
    reranker,
    top_k,
    top_n,
):
    rows = []

    total_start = time.perf_counter()

    for i, item in enumerate(test_data):

        query = item["question"]

        # Ground truth theo mapping đã xác nhận
        relevant_id = f"doc_{i:03d}"

        # =========================
        # Retrieval
        # =========================
        start = time.perf_counter()

        retrieval_result = retriever.retrieve(
            query,
            top_k=top_k
        )

        retrieval_ms = (
            time.perf_counter() - start
        ) * 1000

        candidates = retrieval_result["documents"]

        # =========================
        # Reranking
        # =========================
        start = time.perf_counter()

        rerank_result = reranker.rerank(
            query,
            candidates,
            top_k=top_n
        )

        reranker_ms = (
            time.perf_counter() - start
        ) * 1000

        final_docs = rerank_result["documents"]

        # =========================
        # Find relevant document
        # =========================
        relevant_rank = None

        for result in final_docs:

            if result["document"]["id"] == relevant_id:
                relevant_rank = result["rank"]
                break

        # =========================
        # Metrics
        # =========================
        hit = 1 if relevant_rank is not None else 0

        reciprocal_rank = (
            1 / relevant_rank
            if relevant_rank is not None
            else 0
        )

        total_ms = retrieval_ms + reranker_ms

        rows.append({
            "query_index": i,
            "id": item["id"],
            "question": query,

            "retriever_top_k": top_k,
            "reranker_top_n": top_n,

            "relevant_id": relevant_id,
            "relevant_rank": relevant_rank,

            "hit": hit,
            "rr": reciprocal_rank,

            "retrieval_ms": retrieval_ms,
            "reranker_ms": reranker_ms,
            "total_ms": total_ms,
        })

        # Progress mỗi 100 query
        if (i + 1) % 100 == 0:
            elapsed = time.perf_counter() - total_start

            print(
                f"  {i + 1:4d}/{len(test_data)} "
                f"queries | "
                f"elapsed: {elapsed:.1f}s"
            )

    return rows

In [25]:
all_results = {}

for config in CONFIGS:

    top_k = config["retriever_top_k"]
    top_n = config["reranker_top_n"]

    print("\n" + "=" * 80)
    print(f"Running: Top-{top_k} → Top-{top_n}")
    print("=" * 80)

    start = time.perf_counter()

    rows = evaluate_config(
        test_data=test_data,
        retriever=retriever,
        reranker=reranker,
        top_k=top_k,
        top_n=top_n,
    )

    elapsed = time.perf_counter() - start

    key = f"Top-{top_k} → Top-{top_n}"

    all_results[key] = rows

    print(f"\n✅ Finished: {key}")
    print(f"Total time: {elapsed:.2f} seconds")


Running: Top-5 → Top-3
   100/1003 queries | elapsed: 0.8s
   200/1003 queries | elapsed: 1.6s
   300/1003 queries | elapsed: 2.3s
   400/1003 queries | elapsed: 3.1s
   500/1003 queries | elapsed: 3.7s
   600/1003 queries | elapsed: 4.4s
   700/1003 queries | elapsed: 5.1s
   800/1003 queries | elapsed: 5.8s
   900/1003 queries | elapsed: 6.5s
  1000/1003 queries | elapsed: 7.2s

✅ Finished: Top-5 → Top-3
Total time: 7.21 seconds

Running: Top-10 → Top-3
   100/1003 queries | elapsed: 0.8s
   200/1003 queries | elapsed: 1.7s
   300/1003 queries | elapsed: 2.5s
   400/1003 queries | elapsed: 3.4s
   500/1003 queries | elapsed: 4.3s
   600/1003 queries | elapsed: 5.2s
   700/1003 queries | elapsed: 6.1s
   800/1003 queries | elapsed: 6.8s
   900/1003 queries | elapsed: 7.7s
  1000/1003 queries | elapsed: 8.5s

✅ Finished: Top-10 → Top-3
Total time: 8.54 seconds

Running: Top-20 → Top-5
   100/1003 queries | elapsed: 1.2s
   200/1003 queries | elapsed: 2.3s
   300/1003 queries | elapsed

In [26]:
import pandas as pd

summary_rows = []

for strategy, rows in all_results.items():

    df = pd.DataFrame(rows)

    summary_rows.append({
        "Strategy": strategy,

        "Hit Rate": df["hit"].mean(),

        "MRR": df["rr"].mean(),

        "Avg Retriever (ms)": df["retrieval_ms"].mean(),

        "Avg Reranker (ms)": df["reranker_ms"].mean(),

        "Avg Total (ms)": df["total_ms"].mean(),

        "Median Total (ms)": df["total_ms"].median(),

        "P95 Total (ms)": df["total_ms"].quantile(0.95),
    })

summary_df = pd.DataFrame(summary_rows)

summary_df

,Strategy,Hit Rate,MRR,Avg Retriever (ms),Avg Reranker (ms),Avg Total (ms),Median Total (ms),P95 Total (ms)
0,Top-5 → Top-3,0.653041,0.612496,2.371008,4.815298,7.186306,6.507167,10.659892
1,Top-10 → Top-3,0.725823,0.676471,2.402612,6.109167,8.511779,8.249433,11.907380
2,Top-20 → Top-5,0.788634,0.725955,2.337621,9.318400,11.656021,11.611881,14.706620
3,Top-50 → Top-5,0.841476,0.770754,2.472333,18.432171,20.904504,20.500231,25.779683


In [27]:
import pandas as pd
import time

recall_rows = []

for top_k in [5, 10, 20, 50]:
    hits = 0
    retrieval_times = []

    print("=" * 60)
    print(f"Evaluating Retriever Recall@{top_k}")
    print("=" * 60)

    start_total = time.perf_counter()

    for i, item in enumerate(test_data):
        query = item["question"]
        relevant_id = f"doc_{i:03d}"

        start = time.perf_counter()

        result = retriever.retrieve(
            query,
            top_k=top_k
        )

        retrieval_ms = (time.perf_counter() - start) * 1000
        retrieval_times.append(retrieval_ms)

        retrieved_docs = result["documents"]

        retrieved_ids = {
            doc["id"]
            for doc in retrieved_docs
        }

        if relevant_id in retrieved_ids:
            hits += 1

        if (i + 1) % 100 == 0:
            print(f"{i + 1:4d}/{len(test_data)}")

    elapsed = time.perf_counter() - start_total

    recall = hits / len(test_data)

    recall_rows.append({
        "Top-K": top_k,
        "Retriever Recall@K": recall,
        "Hit Count": hits,
        "Total Queries": len(test_data),
        "Avg Retrieval (ms)": sum(retrieval_times) / len(retrieval_times),
        "Median Retrieval (ms)": pd.Series(retrieval_times).median(),
        "P95 Retrieval (ms)": pd.Series(retrieval_times).quantile(0.95),
        "Total Time (s)": elapsed,
    })

    print(f"\nRecall@{top_k}: {recall:.4f}")
    print(f"Hits: {hits}/{len(test_data)}")
    print(f"Avg retrieval: {recall_rows[-1]['Avg Retrieval (ms)']:.3f} ms")
    print(f"Total time: {elapsed:.2f} s")
    print()

recall_df = pd.DataFrame(recall_rows)

recall_df

Evaluating Retriever Recall@5
 100/1003
 200/1003
 300/1003
 400/1003
 500/1003
 600/1003
 700/1003
 800/1003
 900/1003
1000/1003

Recall@5: 0.6560
Hits: 658/1003
Avg retrieval: 2.135 ms
Total time: 2.15 s

Evaluating Retriever Recall@10
 100/1003
 200/1003
 300/1003
 400/1003
 500/1003
 600/1003
 700/1003
 800/1003
 900/1003
1000/1003

Recall@10: 0.7348
Hits: 737/1003
Avg retrieval: 2.219 ms
Total time: 2.23 s

Evaluating Retriever Recall@20
 100/1003
 200/1003
 300/1003
 400/1003
 500/1003
 600/1003
 700/1003
 800/1003
 900/1003
1000/1003

Recall@20: 0.7926
Hits: 795/1003
Avg retrieval: 2.086 ms
Total time: 2.10 s

Evaluating Retriever Recall@50
 100/1003
 200/1003
 300/1003
 400/1003
 500/1003
 600/1003
 700/1003
 800/1003
 900/1003
1000/1003

Recall@50: 0.8524
Hits: 855/1003
Avg retrieval: 2.103 ms
Total time: 2.12 s



,Top-K,Retriever Recall@K,Hit Count,Total Queries,Avg Retrieval (ms),Median Retrieval (ms),P95 Retrieval (ms),Total Time (s)
0,5,0.656032,658,1003,2.135000,2.079735,2.210306,2.145380
1,10,0.734796,737,1003,2.218572,2.097442,2.455429,2.230107
2,20,0.792622,795,1003,2.086178,2.078735,2.187768,2.098079
3,50,0.852443,855,1003,2.103330,2.093079,2.231755,2.117560


In [28]:
from pathlib import Path
import json

RESULT_DIR = PROJECT_ROOT / "eval" / "reranking_strategy"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Saving results to:")
print(RESULT_DIR)

Saving results to:
/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/reranking_strategy


In [29]:
json_path = RESULT_DIR / "reranking_results.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        all_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved: {json_path}")

Saved: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/reranking_strategy/reranking_results.json


In [30]:
recall_csv_path = RESULT_DIR / "retriever_recall.csv"

recall_df.to_csv(
    recall_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved: {recall_csv_path}")

Saved: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/reranking_strategy/retriever_recall.csv


In [31]:
import pandas as pd

summary_rows = []

for strategy, rows in all_results.items():
    df = pd.DataFrame(rows)

    summary_rows.append({
        "Strategy": strategy,
        "Hit Rate": df["hit"].mean(),
        "MRR": df["rr"].mean(),
        "Avg Retriever (ms)": df["retrieval_ms"].mean(),
        "Avg Reranker (ms)": df["reranker_ms"].mean(),
        "Avg Total (ms)": df["total_ms"].mean(),
        "Median Total (ms)": df["total_ms"].median(),
        "P95 Total (ms)": df["total_ms"].quantile(0.95),
    })

summary_df = pd.DataFrame(summary_rows)

summary_df

,Strategy,Hit Rate,MRR,Avg Retriever (ms),Avg Reranker (ms),Avg Total (ms),Median Total (ms),P95 Total (ms)
0,Top-5 → Top-3,0.653041,0.612496,2.371008,4.815298,7.186306,6.507167,10.659892
1,Top-10 → Top-3,0.725823,0.676471,2.402612,6.109167,8.511779,8.249433,11.907380
2,Top-20 → Top-5,0.788634,0.725955,2.337621,9.318400,11.656021,11.611881,14.706620
3,Top-50 → Top-5,0.841476,0.770754,2.472333,18.432171,20.904504,20.500231,25.779683


In [32]:
summary_csv_path = RESULT_DIR / "reranking_summary.csv"

summary_df.to_csv(
    summary_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved: {summary_csv_path}")

Saved: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/reranking_strategy/reranking_summary.csv
